<a href="https://colab.research.google.com/github/NoiHalldorsson/Identifying-a-Rigid-Body-s-Moment-of-Inertia-Matrix-with-a-Two-Stage-Physics-Informed-Neural-Network/blob/main/PHYS495.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense, Lambda
from tensorflow.keras.layers import Dense, Lambda
import scipy
from scipy.integrate import odeint, solve_ivp
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

In [6]:
tf.compat.v1.enable_eager_execution()
tf.keras.backend.set_floatx('float64')

TOTAL_LOSS_NAME = 'Loss'
# Main trainable neural network body. Follows Keras API
class PINN_model(tf.keras.Model):
    def __init__(self,
                 layer_sizes=[1, 32, 32, 3],
                 activations=[None, "tanh", "tanh", "linear"],
                 kernel_initializer='glorot_normal',
                 improved_MLP=False,
                 parameters=[],
                 **kwargs):
        super().__init__(**kwargs)
        assert len(activations) == len(layer_sizes)
        self.layer_sizes = layer_sizes
        self.activations = activations
        self.input_dim = layer_sizes[0]
        self.output_dim = layer_sizes[-1]
        self.improved_MLP = improved_MLP  # (https://doi.org/10.1137/20M1318043)
        self.parameters = parameters
        # Define MLP
        self.model_layers = [Dense(nu, activation=act, kernel_initializer=kernel_initializer)
                             for nu, act in zip(self.layer_sizes[1:], self.activations[1:])]

        if self.improved_MLP:
            assert len(np.unique(self.layer_sizes[1:-1])) == 1
            self.attention_layers = [Dense(self.layer_sizes[1], activation=self.activations[1],
                                           kernel_initializer=kernel_initializer)
                                           for _ in range(2)]

        self.build(input_shape=(None, self.input_dim))

    def call(self, X):
        """Forward-pass through neural network. Functional API"""
        if (func := self.activations[0]) is not None:  # apply input transformation
          X = tf.keras.activations.get(func)(X)

        if self.improved_MLP:
            U, V = self.attention_layers[0](X), self.attention_layers[1](X)
            UV_transform = Lambda(lambda x: (1.0 - x[0]) * x[1] + x[0] * x[2])

        for layer in self.model_layers:
            X = layer(X)
            if self.improved_MLP:
              X = UV_transform([X, U, V])
        return X

    def loss_function(self, XY_data, loss_weights, loss_funcs):
        losses = {}

        # Loss calculation term-by-term
        for term_name, data_args in XY_data.items():
            assert len(data_args) in (2, 3)  # 2 means (X, Y), 3: (X, Y, func)
            if len(data_args) == 2:
                xi, yi = data_args
                model_func = self  # if no func given, then just model output (for typical IC/BC)
            else:
                xi, yi, func = data_args  # if func is given, then apply it
                model_func = lambda z: func(self, z)  # needed for PDE, and other Neumann-type BC/IC
            wi = loss_weights.get(term_name, 1.0)  # default term weight
            loss_func = loss_funcs.get(term_name, tf.losses.MeanSquaredError())  # default MSE loss type

            yi_model = model_func(xi)  # calculate function on model (PDE, BC/IC, other)
            yi = tf.cast(yi, self.dtype)  # preserve dtype of model
            yi_model = tf.cast(yi_model, dtype=self.dtype)  # preserve dtype of model

            losses[term_name] = wi * loss_func(yi, yi_model)
        # losses['Reg'] = 1e-6 * tf.add_n([tf.reduce_sum(v**2) for v in [a, b, c]])
        losses[TOTAL_LOSS_NAME] = tf.reduce_sum(list(losses.values()))
        return losses

    @tf.function
    def train_step(self, optimizer, XY_data, loss_weights, loss_funcs):
        # Compute current loss and gradient wrt parameters
        with tf.GradientTape() as tape:
            # tape.watch(trainable_vars)
            losses = self.loss_function(XY_data, loss_weights, loss_funcs)

        trainable_vars = tuple(self.trainable_variables) + tuple(self.parameters)
        grad = tape.gradient(losses[TOTAL_LOSS_NAME], trainable_vars)
        # Perform gradient descent step
        optimizer.apply_gradients(zip(grad, trainable_vars))
        return losses, grad, trainable_vars

    def train(self, optimizer, XY_data, loss_weights=None, loss_funcs=None,
              num_epochs=10000, num_batches=1, callbacks=None, progress_bar=True,
              print_every=1000, print_loss_terms=False, print_parameters=False):

        # Check arguments
        callbacks = [] if callbacks is None else callbacks
        assert isinstance(XY_data, dict), "`XY_data` must be given as dict"
        loss_weights = {} if loss_weights is None else loss_weights
        assert isinstance(loss_weights, dict), "`loss_weights` must be given as dict"
        loss_funcs = {} if loss_funcs is None else loss_funcs
        assert isinstance(loss_funcs, dict) or loss_funcs is None, "`loss_funcs` must be given as dict"

        # Create history object that stores losses and params over epochs
        history = {TOTAL_LOSS_NAME: []}
        history.update({term_name: [] for term_name in XY_data.keys()})

        term_names = list(history.keys())[1:]
        param_names = [param.name.split(':')[0] for param in self.parameters]

        history.update({name: [] for name in param_names})

        ## Loop through callbacks (on_train_begin)
        for callback in callbacks:
          callback.on_train_begin(self, history)

        # Get all trainable variables including model weights and physics parameters
        # all_trainable_vars = self.trainable_variables + self.parameters

        # Main train loop
        for i in tqdm(range(num_epochs), display=progress_bar):  # Loop over epochs
          ## Loop through callbacks (on_epoch_begin)
          for callback in callbacks:
            callback.on_epoch_begin(self, history)

          for name in history.keys():
            history[name].append(0.0)

          ## Loop over batches within single epoch
          for XY_batch in batch_generator(XY_data, num_batches=num_batches):
            losses_batch, grad_theta, train_vars1 = self.train_step(optimizer, XY_batch,
                                           loss_weights=loss_weights,
                                           loss_funcs=loss_funcs,
                                           )

            ## Write the loss values and parameters (average over batches)
            for term_name, term_val in losses_batch.items():
              history[term_name][i] += term_val.numpy() / num_batches  # average
            for name, param in zip(param_names, self.parameters):
              history[name][i] += param.numpy() / num_batches  # average

          ## Loop through callbacks (on_epoch_end)
          for callback in callbacks:
            callback.on_epoch_end(self, history)

          ## Print status
          if print_every and ((i + 1) % print_every == 0):
            print_stats = [f"Epoch {i + 1}:",
                           f"{TOTAL_LOSS_NAME} {history[TOTAL_LOSS_NAME][i]:.5g}"]
            if print_loss_terms:
              print_stats.extend([f"{name} {history[name][i]:.5g}" for name in term_names])
            if print_parameters:
              print_stats.extend([f"{name} {history[name][i]:.5g}" for name in param_names])

            print_stats = '    '.join(print_stats)
            print(print_stats)

        ## Loop through callbacks (on_train_end)
        for callback in callbacks:
          callback.on_train_end(self, history)

        return history


def batch_generator(XY_data, num_batches=4):
  assert isinstance(XY_data, dict)
  batch_axis = 0
  batch_inds = {}

  # Make batches
  for term_name, (xi, yi, *func) in XY_data.items():
    inds = np.arange(xi.shape[batch_axis])
    np.random.shuffle(inds)  # shuffle indices
    batch_inds[term_name] = np.array_split(inds, num_batches, axis=batch_axis)

  # Sample batches sequentially from batch indices
  for i in range(num_batches):
    xy_batch = {term_name: (np.take(xi, batch_inds[term_name][i], axis=batch_axis),
                            np.take(yi, batch_inds[term_name][i], axis=batch_axis),
                            *func)
                for term_name, (xi, yi, *func) in XY_data.items()}

    yield xy_batch


class PINN_callback:
  def on_train_begin(self, model, history):
    pass

  def on_epoch_begin(self, model, history):
    pass

  def on_epoch_end(self, model, history):
    pass

  def on_train_end(self, model, history):
    pass


class BestWeights(PINN_callback):
    """Callback to save best weights of the model.

    Args:
        monitor: Quantity to be monitored ('loss' or 'val_loss'). By default, 'loss'
        start_monitor: below which 'monitor' value callback starts watching 'monitor' and saves best weights
        freq: how often save weights
        verbose: verbosity mode (0 or 1).
    """

    def __init__(self,
                 monitor='Loss',
                 start_monitor=1e-2,
                 freq=1,
                 verbose=1,
                 ):

        self.monitor = monitor
        self.verbose = verbose
        self.freq = freq
        self.start_monitor = start_monitor
        self.best_monitor = np.inf
        self.best_epoch = None
        self.best_weights = None

    def on_train_begin(self, model, history):
        # Allow instances to be re-used
        self.best_monitor = np.inf
        self.best_epoch = None
        self.best_weights = None

    def on_epoch_end(self, model, history):
        current = history[self.monitor][-1]
        epoch = len(history[self.monitor])
        if current is None:
            return
        if (epoch > 0) and (epoch % self.freq == 0) and \
           (current < self.best_monitor) and (current < self.start_monitor):
            self.best_monitor = current
            self.best_epoch = epoch
            self.best_weights = model.get_weights()

            if self.verbose:
                print('Epoch %05d: saving best weights' % (epoch + 1))
                print(f'{self.monitor}: {self.best_monitor:.3g}')

    def on_train_end(self, model, history):
        if self.best_weights is not None:
            model.set_weights(self.best_weights)
        print(f"Set best weights with {self.monitor} = {self.best_monitor:.3g}")

# Euler Equation PINN

In [7]:
I1_star, I2_star, I3_star, a_star, b_star, c_star = 1.,2.,3.,0.5,0.7,0.2
tmin, tmax = 0.0, 1.

In [ ]:
raw_L11 = tf.Variable(1., trainable=True, name='L11')
raw_L22 = tf.Variable(1., trainable=True, name='L22')
raw_L33 = tf.Variable(1., trainable=True, name='L33')
raw_L21 = tf.Variable(1., trainable=True, name='L21')
raw_L31 = tf.Variable(1., trainable=True, name='L31')
raw_L32 = tf.Variable(1., trainable=True, name='L32')
eps = 1e-6
rng = np.random.default_rng(42)
N = 5
t_k = np.sort(rng.uniform(0, 1, size=N))
J_raw = rng.normal(size=(N, 3))
J_k = J_raw / np.linalg.norm(J_raw, axis=1, keepdims=True)
t_k_np = t_k
J_k_np = J_k
def lower_tri_matrix(dtype):
    L11 = tf.nn.softplus(tf.cast(raw_L11, dtype=dtype)) + eps
    L22 = tf.nn.softplus(tf.cast(raw_L22, dtype=dtype)) + eps
    L33 = tf.nn.softplus(tf.cast(raw_L33, dtype=dtype)) + eps
    L21 = tf.cast(raw_L21, dtype=dtype)
    L31 = tf.cast(raw_L31, dtype=dtype)
    L32 = tf.cast(raw_L32, dtype=dtype)

    zeros = tf.zeros([], dtype=dtype)
    L = tf.stack([
        tf.stack([L11, zeros, zeros]),
        tf.stack([L21, L22, zeros]),
        tf.stack([L31, L32, L33 ]),])
    return L

def make_tau_fn(dtype):
    t_k = tf.constant(t_k_np, dtype=dtype)      # (N_imp,)
    J_k = tf.constant(J_k_np, dtype=dtype)      # (N_imp, 3)

    c = tf.constant(5.0/np.sqrt(np.pi), dtype=dtype)
    ten = tf.constant(10.0, dtype=dtype)

    def tau_of_t(t):  # t: (batch,1) or (batch,)
        t = tf.cast(t, dtype)
        t = tf.reshape(t, (-1,))                # (batch,)
        dt = t[:, None] - t_k[None, :]          # (batch, N_imp)
        w  = c * tf.exp(-(ten*dt)**2)           # (batch, N_imp)
        # (batch, N_imp) @ (N_imp, 3) -> (batch, 3)
        return tf.linalg.matmul(w, J_k)
    return tau_of_t

def Euler_residual(model, t):
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(t)
        Om = model(t)
        dOm = tf.squeeze(tape.batch_jacobian(Om, t), axis=-1)
        dOm = tf.cast(dOm, dtype=model.dtype)
    del tape
    L = lower_tri_matrix(Om.dtype)
    I = tf.matmul(L, L, transpose_b=True)
    I_mat = I
    tau_fn = make_tau_fn(Om.dtype)
    tau = tf.cast(tau_fn(t), Om.dtype)
    H = tf.einsum('ij,nj->ni', I_mat, Om)
    IdOm = tf.einsum('ij,nj->ni', I_mat, dOm)
    IdOm = tf.cast(IdOm, Om.dtype)
    cross = tf.linalg.cross(Om, H)
    return tf.cast(IdOm + cross - tau, Om.dtype)


def Energy_residual(model, t):
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(t)
        Om  = model(t)                                   # (N,3)
        dOm = tf.squeeze(tape.batch_jacobian(Om, t), -1) # (N,3)
    del tape
    L = lower_tri_matrix(Om.dtype)
    I = tf.matmul(L, L, transpose_b=True)
    IdOm   = tf.einsum('ij,nj->ni', I, dOm)             # (N,3)
    tau    = tf.cast(make_tau_fn(Om.dtype)(t), Om.dtype) # (N,3)

    OmIdOm = tf.reduce_sum(Om * IdOm, axis=-1)          # (N,)
    OmTau  = tf.reduce_sum(Om * tau,  axis=-1)          # (N,)
    return OmIdOm - OmTau
"""
def AngMom_residual(model, t):
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(t)
        Om = model(t)
        dOm = tf.squeeze(tape.batch_jacobian(Om, t), axis=-1)
    L = lower_tri_matrix(Om.dtype)
    I = tf.matmul(L, L, transpose_b=True)
    I_mat = I
    tau_fn = make_tau_fn(Om.dtype)
    tau = tf.cast(tau_fn(t), Om.dtype)
    term1 = tf.einsum('ij,nj->ni', I_mat, Om)
    term1 = tf.cast(term1, Om.dtype)
    IdOm = tf.einsum('ij,nj->ni', I_mat, dOm)
    IdOm = tf.cast(IdOm, Om.dtype)
    IOm = tf.einsum('ij,nj->ni', I_mat, Om)
    IOm = tf.cast(IOm, Om.dtype)
    return 2*(tf.reduce_sum(IdOm*IOm, axis=-1)-tf.reduce_sum(term1*tau, axis=-1))
"""
def Euler_data(t, Om, tau_func):
    Om = np.asarray(Om, dtype=np.float64) # teriyaki chicken

    I_true = np.array([[I1_star, a_star, b_star],
                       [a_star, I2_star, c_star],
                       [b_star, c_star, I3_star]], dtype=np.float64)

    I_Om = I_true @ Om
    tau_t = tau_func(t)
    rhs  = -np.cross(Om, I_Om) + tau_t
    dOm  = np.linalg.solve(I_true, rhs)
    return dOm

"Here is the torque experienced by the body at short impulses"
def make_tau_func(t_k, J_k):
    # returns a function tau(t) -> (3,)
    def tau_of_t(t):
        dt = t - t_k                      # shape (N,)
        w = (5/np.sqrt(np.pi)) * np.exp(-(10*dt)**2)   # (N,)
        return (w[:, None] * J_k).sum(axis=0)          # (3,)
    return tau_of_t
rng = np.random.default_rng(42)
N = 5
t_k = np.sort(rng.uniform(0, 1, size=N))
J_raw = rng.normal(size=(N, 3))
J_k = J_raw / np.linalg.norm(J_raw, axis=1, keepdims=True)
tau_func = make_tau_func(t_k, J_k)
"End of torque definition"


# Y_eq 512x2 matrix of zeroes since both the equations in Euler_equation equals zero
T_eq = np.random.uniform(tmin, tmax, (512,1))
Y_eq = np.zeros((T_eq.shape[0], 3),dtype=np.float64)

Y_eq2 = np.zeros((T_eq.shape[0], 1),dtype=np.float64)


T_ic = np.zeros((8, 1),dtype=np.float64)
Y_ic = np.tile(np.array([[0.8, -1.2, 0.5]], dtype=np.float64), (T_ic.shape[0], 1))

T_point_set = np.random.uniform(tmin, tmax, (64, 1))
T_point_set_sorted = np.sort(T_point_set, axis=0)
# need to flatten T_point_set_sorted for solve_ivp
# we will use the 64x1 array for the dictionary of points
T_eval = T_point_set_sorted.flatten()
Om1_0, Om2_0, Om3_0 = 0.8, -1.2, 0.5
sol = solve_ivp(Euler_data, [0,tmax], [Om1_0,Om2_0,Om3_0], t_eval=T_eval, args=(tau_func,))
T_sol = sol.t.reshape(-1, 1)
Y_point_set = sol.y.T


XY_data = {"Equation": (T_eq, Y_eq, Euler_residual),
           "IC": (T_ic, Y_ic),
           "PointSet": (T_sol, Y_point_set),
           "Energy": (T_eq, Y_eq2, Energy_residual)}

"""
XY_data = {"Equation": (T_eq, Y_eq, Euler_residual),
           "IC": (T_ic, Y_ic),
           "PointSet": (T_sol, Y_point_set)}
"""
loss_weights = {"Equation": 50.0, "IC": 1.0, "PointSet": 5.0, "Energy": 1.0}


# First activation is now None as we dont have an exact solution
Euler_PINN = PINN_model(layer_sizes=[1, 32, 32, 3],
                     activations=[None, 'tanh', 'tanh', 'linear'],
                     kernel_initializer='glorot_normal',
                     parameters=[raw_L11, raw_L22, raw_L33, raw_L21, raw_L31, raw_L32])

# optimizer = tf.optimizers.Adam(learning_rate=5e-4)
# optimizer.set_lr

boundaries = [20_000, 50_000]  # change at these steps
values     = [1e-3, 5e-4, 1e-4]

lr_schedule = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
    boundaries=boundaries,
    values=values
)

# 2) Use RMSProp with the schedule
optimizer = tf.keras.optimizers.RMSprop(
    learning_rate=lr_schedule,   # schedule, not a scalar
    rho=0.9,                     # default momentum term
    epsilon=1e-7                 # TF default; tune if needed
)

history = Euler_PINN.train(optimizer,
                        XY_data=XY_data,
                        loss_weights=loss_weights,
                        loss_funcs=None,  # default all MSE terms
                        num_epochs=150000,
                        num_batches=1,
                        callbacks=None,
                        print_every=1000,
                        print_loss_terms=True,
                        print_parameters=True)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'pinn_model', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


  0%|          | 0/150000 [00:00<?, ?it/s]

Epoch 1000:    Loss 33.837    Equation 25.593    IC 0.22447    PointSet 5.5402    Energy 2.4791    L11 1.1418    L22 0.80103    L33 1.0229    L21 1.0032    L31 0.92851    L32 1.1472
Epoch 2000:    Loss 27.666    Equation 20.393    IC 0.19422    PointSet 5.3133    Energy 1.7653    L11 1.2559    L22 0.75694    L33 1.0562    L21 0.99513    L31 0.93457    L32 1.1643
Epoch 3000:    Loss 16.813    Equation 12.798    IC 0.2823    PointSet 3.0385    Energy 0.69425    L11 1.3105    L22 0.94334    L33 1.1342    L21 0.96114    L31 0.97012    L32 1.063
Epoch 4000:    Loss 11.127    Equation 9.434    IC 0.18086    PointSet 1.0791    Energy 0.43342    L11 1.2445    L22 1.0557    L33 1.1857    L21 0.94244    L31 0.92103    L32 0.99791
Epoch 5000:    Loss 4.0124    Equation 3.6992    IC 0.031757    PointSet 0.2009    Energy 0.080638    L11 1.1384    L22 1.0131    L33 1.2483    L21 0.98513    L31 0.82945    L32 0.85437
Epoch 6000:    Loss 2.4871    Equation 2.2564    IC 0.028139    PointSet 0.14809    

In [ ]:
L11, L22, L33, L21, L31, L32 = [v.numpy() for v in [raw_L11, raw_L22, raw_L33, raw_L21, raw_L31, raw_L32]]
I11 = L11**2
I22 = L21**2 + L22**2
I33 = L31**2 + L32**2 + L33**2
I12 = L11*L21
I13 = L11*L31
I23 = L21*L31 + L22*L32
print(I11, I22, I33, I12, I13, I23)

'''
print('The found value of I1 is:',Euler_PINN.parameters[0].numpy()**2,' True value:',I1_star)
print('The found value of I2 is:',Euler_PINN.parameters[1].numpy()**2+Euler_PINN.parameters[3].numpy()**2,' True value:',I2_star)
print('The found value of I3 is:',Euler_PINN.parameters[2].numpy()**2+Euler_PINN.parameters[4].numpy()**2+Euler_PINN.parameters[5].numpy()**2,' True value:',I3_star)
print('The found value of a is:',Euler_PINN.parameters[0].numpy()*Euler_PINN.parameters[3].numpy(),' True value:',a_star)
print('The found value of b is:',Euler_PINN.parameters[0].numpy()*Euler_PINN.parameters[4].numpy(),' True value:',b_star)
print('The found value of c is:',Euler_PINN.parameters[4].numpy()*Euler_PINN.parameters[3].numpy()+Euler_PINN.parameters[1].numpy()*Euler_PINN.parameters[5].numpy(),' True value:',c_star)
raw_L11, raw_L22, raw_L33, raw_L21, raw_L31, raw_L32
plt.plot(history['Loss'])
plt.title("Neural Network Loss vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.yscale('log')
plt.show()
'''


In [ ]:
t_pred = np.linspace(0, tmax, 1000).reshape(-1, 1)
y_pred = Euler_PINN.predict(t_pred)

plt.figure(figsize=(7, 4))

plt.plot(t_pred, y_pred, lw=3)
plt.plot(T_sol, Y_point_set, 'o', ms=4)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
N = 5
t_k = np.sort(rng.uniform(0, 10, size=N))
t = np.linspace(0, 10, 5000)
J_raw = rng.normal(size=(N, 3))
J_k = J_raw / np.linalg.norm(J_raw, axis=1, keepdims=True)  # shape (N,3)

dt = t[:, None] - t_k[None, :]                  # (M,N)
K = (5/np.sqrt(np.pi)) * np.exp(-(10*dt)**2)     # (M,N)  ← use np.exp, not np.e**

tau = K @ J_k

print(tau)                                  # (M,3)

plt.plot(t, tau[:,0], label='x')
plt.plot(t, tau[:,1], label='y')
plt.plot(t, tau[:,2], label='z')
for tk in t_k: plt.axvline(tk, ls='--', lw=0.8)
plt.legend(); plt.xlabel('t'); plt.ylabel('τ components'); plt.tight_layout(); plt.show()

In [ ]:
"""
Physics-Informed Neural Network (PINN) for Euler's Rigid Body Equations
------------------------------------------------------------------------
Solves and identifies the 3x3 symmetric moment of inertia matrix I (6 unknowns)
from synthetic angular velocity data Omega(t) and its time derivative dOmega/dt,
assuming zero external torque:

    I dOmega/dt + Omega x (I Omega) = 0

- Synthetic data is generated by simulating the ODE with a *ground-truth* SPD inertia matrix.
- The PINN learns Omega(t) and the unknown inertia parameters simultaneously.
- TensorFlow 2.x (eager mode) is used. No SciPy required.

How to run:
    python pinn_euler_rigidbody_tf.py

This script will:
  1) Generate synthetic (t, Omega(t), dOmega/dt) data
  2) Train the PINN to match data and satisfy physics
  3) Print the identified inertia matrix and compare to ground truth

Notes:
- Inertia matrix is parameterized via a Cholesky factor L with positive diagonals
  (I = L L^T), guaranteeing symmetry and positive definiteness with 6 parameters.
- You can tweak hyperparameters in the CONFIG section.
"""
from __future__ import annotations
import os
import math
import time
import numpy as np
import tensorflow as tf

# ---------------------------
# CONFIG
# ---------------------------
SEED = 1234
T_FINAL = 5.0          # total simulation time (s)
N_STEPS = 1001         # number of time samples (>= 2)
NOISE_STD = 0.0        # optional Gaussian noise on Omega data (rad/s)

# Training hyperparameters
EPOCHS = 6000
LR = 1e-3
PRINT_EVERY = 500

# Loss weights
W_DATA = 1.0           # weight for Omega(t) data MSE
W_DERIV = 0.2          # weight for dOmega/dt data MSE
W_PHYS = 10.0          # weight for physics residual MSE

# Network architecture
HIDDEN_LAYERS = 4
HIDDEN_UNITS = 64
ACTIVATION = "tanh"    # good default for PINNs

# Minimum positive diagonal for Cholesky to ensure numerical stability
MIN_DIAG = 1e-3

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---------------------------
# Utility functions
# ---------------------------

def cross_batch(a: tf.Tensor, b: tf.Tensor) -> tf.Tensor:
    """Batched cross product for tensors of shape (N, 3).
    Returns shape (N, 3).
    """
    ax, ay, az = a[:, 0], a[:, 1], a[:, 2]
    bx, by, bz = b[:, 0], b[:, 1], b[:, 2]
    cx = ay * bz - az * by
    cy = az * bx - ax * bz
    cz = ax * by - ay * bx
    return tf.stack([cx, cy, cz], axis=1)


def softplus_positive(x: tf.Tensor, min_val: float = MIN_DIAG) -> tf.Tensor:
    """Ensure strictly positive values for Cholesky diagonals."""
    return tf.math.softplus(x) + min_val


# ---------------------------
# Ground-truth inertia (SPD)
# ---------------------------

def sample_spd_inertia(seed: int = SEED) -> np.ndarray:
    """Create a random but well-conditioned SPD inertia matrix via Cholesky.
    Return I_true as (3, 3) numpy array.
    """
    rng = np.random.default_rng(seed)
    # Create a lower-triangular L with positive diagonals.
    # Choose diagonals to be of order ~ 1 and off-diagonals smaller.
    a = 0.9 + 0.4 * rng.random()    # > 0
    c = 1.0 + 0.4 * rng.random()
    f = 1.1 + 0.4 * rng.random()
    b = 0.05 * rng.normal()
    d = 0.05 * rng.normal()
    e = 0.05 * rng.normal()
    L = np.array([[a, 0.0, 0.0],
                  [b, c,   0.0],
                  [d, e,   f  ]], dtype=np.float64)
    I_true = L @ L.T
    return I_true


# ---------------------------
# ODE: dOmega/dt = - I^{-1} (Omega x (I Omega))
# ---------------------------

def rhs_omega(t: float, omega: np.ndarray, I: np.ndarray) -> np.ndarray:
    """Right-hand side of Euler's free rigid-body equation.
    omega: shape (3,)
    I: SPD inertia matrix (3x3)
    return dOmega/dt as (3,)
    """
    Iomega = I @ omega
    cross_term = np.cross(omega, Iomega)
    domega = - np.linalg.solve(I, cross_term)
    return domega


def rk4_step(fun, t, y, h, I) -> np.ndarray:
    """Classic RK4 integrator step."""
    k1 = fun(t, y, I)
    k2 = fun(t + 0.5 * h, y + 0.5 * h * k1, I)
    k3 = fun(t + 0.5 * h, y + 0.5 * h * k2, I)
    k4 = fun(t + h, y + h * k3, I)
    return y + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)


def simulate_omega(I_true: np.ndarray,
                   T: float = T_FINAL,
                   N: int = N_STEPS,
                   seed: int = SEED) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Simulate Omega(t) over [0, T] with N points using RK4.

    Returns (t, Omega, dOmega_dt) where:
      - t has shape (N,)
      - Omega has shape (N, 3)
      - dOmega_dt has shape (N, 3)
    """
    rng = np.random.default_rng(seed + 1)
    # Random-ish initial angular velocity
    omega0 = rng.uniform(low=-1.0, high=1.0, size=(3,))

    t = np.linspace(0.0, T, N, dtype=np.float64)
    h = t[1] - t[0]

    omega = np.zeros((N, 3), dtype=np.float64)
    domega = np.zeros((N, 3), dtype=np.float64)

    omega[0] = omega0
    domega[0] = rhs_omega(t[0], omega0, I_true)

    for i in range(N - 1):
        w_next = rk4_step(rhs_omega, t[i], omega[i], h, I_true)
        omega[i + 1] = w_next
        domega[i + 1] = rhs_omega(t[i + 1], w_next, I_true)

    return t, omega, domega


# ---------------------------
# PINN Model
# ---------------------------
class OmegaPINN(tf.keras.Model):
    def __init__(self, layers: int, units: int, activation: str = "tanh"):
        super().__init__()
        self.hidden = [tf.keras.layers.Dense(units, activation=activation) for _ in range(layers)]
        self.out = tf.keras.layers.Dense(3, activation=None)

    def call(self, t_in: tf.Tensor) -> tf.Tensor:
        # Expect t_in shape (N, 1)
        z = t_in
        for layer in self.hidden:
            z = layer(z)
        return self.out(z)


class InertiaCholesky(tf.Module):
    """Cholesky-parameterized SPD inertia matrix.

    L = [[a, 0, 0],
         [b, c, 0],
         [d, e, f]] with a,c,f > 0 via softplus.
    """
    def __init__(self):
        super().__init__()
        # Raw variables (unconstrained); diagonals made positive via softplus
        self.a_raw = tf.Variable(0.5, dtype=tf.float64)
        self.c_raw = tf.Variable(0.5, dtype=tf.float64)
        self.f_raw = tf.Variable(0.5, dtype=tf.float64)
        self.b = tf.Variable(0.0, dtype=tf.float64)
        self.d = tf.Variable(0.0, dtype=tf.float64)
        self.e = tf.Variable(0.0, dtype=tf.float64)

    def L(self) -> tf.Tensor:
        a = tf.cast(softplus_positive(self.a_raw), tf.float64)
        c = tf.cast(softplus_positive(self.c_raw), tf.float64)
        f = tf.cast(softplus_positive(self.f_raw), tf.float64)
        b = self.b
        d = self.d
        e = self.e
        return tf.stack([
            tf.stack([a, 0.0, 0.0], axis=0),
            tf.stack([b,   c,  0.0], axis=0),
            tf.stack([d,   e,    f], axis=0)
        ], axis=0)  # shape (3,3)

    def I(self) -> tf.Tensor:
        L = self.L()
        return tf.matmul(L, tf.transpose(L))  # I = L L^T, shape (3,3)

    def trainable_variables(self):  # so optimizers see these
        return [self.a_raw, self.c_raw, self.f_raw, self.b, self.d, self.e]


# ---------------------------
# Training setup
# ---------------------------

def to_tensor(x: np.ndarray, dtype=tf.float64) -> tf.Tensor:
    return tf.convert_to_tensor(x, dtype=dtype)


def make_dataset(I_true_np: np.ndarray):
    t_np, omega_np, domega_np = simulate_omega(I_true_np, T_FINAL, N_STEPS, SEED)
    if NOISE_STD > 0:
        omega_np = omega_np + np.random.normal(scale=NOISE_STD, size=omega_np.shape)
        # derivative noise is often high-variance; keep it clean unless needed
    return t_np, omega_np, domega_np


def build_model_and_params():
    model = OmegaPINN(HIDDEN_LAYERS, HIDDEN_UNITS, activation=ACTIVATION)
    # Build once to set dtypes to float64 for stability
    model.build(input_shape=(None, 1))
    # Switch model to float64
    for v in model.variables:
        v.assign(tf.cast(v, tf.float64))
    inertia = InertiaCholesky()
    return model, inertia


def physics_residual(omega_pred: tf.Tensor,
                     domega_pred: tf.Tensor,
                     I_mat: tf.Tensor) -> tf.Tensor:
    """Compute residual r = I dOmega/dt + Omega x (I Omega) for each sample.
    omega_pred, domega_pred: shape (N, 3)
    I_mat: shape (3, 3)
    return residuals shape (N, 3)
    """
    Iomega = tf.einsum('ij,nj->ni', I_mat, omega_pred)          # (N,3)
    term1 = tf.einsum('ij,nj->ni', I_mat, domega_pred)          # (N,3) = I dOmega/dt
    term2 = cross_batch(omega_pred, Iomega)                     # (N,3) = Omega x (I Omega)
    return term1 + term2


def compute_derivative(model: tf.keras.Model, t_batch: tf.Tensor) -> tf.Tensor:
    """Use auto-diff to compute dOmega/dt for batch t.
    Returns shape (N, 3).
    """
    with tf.GradientTape() as tape:
        tape.watch(t_batch)
        omega_pred = model(t_batch)
    # Full per-sample jacobian: (N, 3, N, 1)
    J = tape.jacobian(omega_pred, t_batch)
    # Extract per-sample diagonal dOmega/dt: (N, 3)
    d_omega_dt = tf.squeeze(tf.linalg.diag_part(J), axis=-1)
    return d_omega_dt


def train(model: tf.keras.Model,
          inertia: InertiaCholesky,
          t_np: np.ndarray,
          omega_np: np.ndarray,
          domega_np: np.ndarray):
    optimizer = tf.keras.optimizers.Adam(LR)

    # Tensors (float64 for stability)
    t_all = to_tensor(t_np.reshape(-1, 1))
    omega_all = to_tensor(omega_np)
    domega_all = to_tensor(domega_np)

    # Warm-up a forward pass to ensure dtypes align
    _ = model(t_all)

    @tf.function
    def train_step():
        with tf.GradientTape(persistent=True) as tape:
            # Predictions
            omega_pred = model(t_all)                              # (N,3)
            # Auto-diff derivative wrt time
            J = tape.jacobian(omega_pred, t_all)                   # (N,3,N,1)
            d_omega_dt = tf.squeeze(tf.linalg.diag_part(J), -1)    # (N,3)

            # Construct inertia matrix from parameters
            I_mat = inertia.I()                                    # (3,3)

            # Physics residuals
            r = physics_residual(omega_pred, d_omega_dt, I_mat)    # (N,3)

            # Losses
            loss_data = tf.reduce_mean(tf.square(omega_pred - omega_all))
            loss_deriv = tf.reduce_mean(tf.square(d_omega_dt - domega_all))
            loss_phys = tf.reduce_mean(tf.square(r))
            loss_total = W_DATA * loss_data + W_DERIV * loss_deriv + W_PHYS * loss_phys

        # Gradients
        vars_all = model.trainable_variables + inertia.trainable_variables()
        grads = tape.gradient(loss_total, vars_all)
        optimizer.apply_gradients(zip(grads, vars_all))

        return loss_total, loss_data, loss_deriv, loss_phys

    for epoch in range(1, EPOCHS + 1):
        total, l_data, l_deriv, l_phys = train_step()
        if epoch % PRINT_EVERY == 0 or epoch == 1:
            print(f"Epoch {epoch:5d} | Total {total.numpy():.6e} | "
                  f"Data {l_data.numpy():.6e} | Deriv {l_deriv.numpy():.6e} | Phys {l_phys.numpy():.6e}")


# ---------------------------
# Main
# ---------------------------
if __name__ == "__main__":
    tf.keras.backend.set_floatx('float64')  # better numerical stability for PINNs

    # 1) Ground truth inertia and synthetic data
    I_true_np = sample_spd_inertia(SEED)
    t_np, omega_np, domega_np = make_dataset(I_true_np)

    print("Ground-truth inertia matrix I_true:\n", I_true_np)

    # 2) Build PINN and train
    model, inertia = build_model_and_params()
    train(model, inertia, t_np, omega_np, domega_np)

    # 3) Results
    I_est_tf = inertia.I()
    I_est = I_est_tf.numpy()

    print("\nEstimated inertia matrix I_est:\n", I_est)

    # Relative error metrics
    rel_err_mat = np.linalg.norm(I_est - I_true_np) / np.linalg.norm(I_true_np)
    print(f"\nRelative Frobenius error ||I_est - I_true|| / ||I_true|| = {rel_err_mat:.6e}")

    # Optionally print element-wise comparison
    def fmt(x):
        return " ".join(f"{v: .5f}" for v in x)

    print("\nI_true (rows):")
    for r in I_true_np:
        print(fmt(r))

    print("\nI_est  (rows):")
    for r in I_est:
        print(fmt(r))

    # Quick check of physics residual on trained model
    t_tf = tf.convert_to_tensor(t_np.reshape(-1, 1), dtype=tf.float64)
    with tf.GradientTape() as tape:
        tape.watch(t_tf)
        omega_pred = model(t_tf)
    J = tape.jacobian(omega_pred, t_tf)
    d_omega_dt = tf.squeeze(tf.linalg.diag_part(J), -1)
    resid = physics_residual(omega_pred, d_omega_dt, inertia.I())
    resid_norm = tf.reduce_mean(tf.norm(resid, axis=1))
    print(f"\nMean residual norm after training: {resid_norm.numpy():.6e}")
